In [1]:
from tqdm import tqdm

import glob, os
import numpy as np
import torch

from IPython.display import clear_output

DEVICE = torch.device("mps") #torch.device("mps")
BATCH_SIZE = 64
NUM_WINDOWS = 200
NUM_EPOCHS = 10

In [8]:
from features.make_features import loaders, detrend_3d, norm_adj
from models.dyn_model import GraphTemporal
from train import train_model, print_acc

In [4]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

DEVICE = torch.device("mps") #torch.device("mps")
gt_operators = ["A3TGCN", "GCLSTM", "TGCN", "DCRNN", "GConvGRU"]
for op in gt_operators:
    model = GraphTemporal(num_ch=19, num_t=NUM_WINDOWS, op=op).to(DEVICE)
    print(op, ": ", count_parameters(model))

A3TGCN :  31679
GCLSTM :  35383
TGCN :  31479
DCRNN :  53655
GConvGRU :  36855


In [9]:
"""Coherence connectivity"""
train_X = np.load("../tuh_data/train_X.npy", mmap_mode="c").astype(np.float16)
train_y = np.load("../tuh_data/train_y.npy", mmap_mode="c").astype(np.float16)
train_graphs = np.load("../tuh_data/train_graphs_coh.npy", mmap_mode="c").astype(np.float16)
test_y = np.load("../tuh_data/test_y.npy", mmap_mode="c").astype(np.float16)
test_X = np.load("../tuh_data/test_X.npy", mmap_mode="c").astype(np.float16)
test_graphs = np.load("../tuh_data/test_graphs_coh.npy", mmap_mode="c").astype(np.float16)
train_X = np.moveaxis(train_X, 1, 2)
test_X = np.moveaxis(test_X, 1, 2)

train_graphs, test_graphs = norm_adj(train_graphs, test_graphs)
train_iter, test_iter  = loaders(train_X, train_graphs, train_y, test_X, test_graphs, test_y, DEVICE, BATCH_SIZE, NUM_WINDOWS)

/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:172: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:278.)
  y = torch.tensor([y], dtype=torch.float).to(device)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:205: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:206: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_iter = DataLoader(test_dataset, 

In [10]:
from sklearn.model_selection import train_test_split

DEVICE = torch.device("mps") #torch.device("cpu")

for idx in range(5): #run 5 times
    train_X_, val_X, _, _ = train_test_split(train_X, train_y, random_state=idx, test_size=0.05)
    train_graphs_, val_graphs, train_y_, val_y = train_test_split(train_graphs, train_y, random_state=idx, test_size=0.05)
    train_iter, val_iter  = loaders(train_X_, train_graphs_, train_y_, val_X, val_graphs, val_y, DEVICE, BATCH_SIZE, NUM_WINDOWS)
    gt_operators = ["DCRNN", "A3TGCN", "GCLSTM", "TGCN", "GConvGRU"]
    for op in gt_operators:
        model = GraphTemporal(num_ch=19, num_t=NUM_WINDOWS, op=op).to(DEVICE)
        model = train_model(model, NUM_EPOCHS, train_iter, val_iter)
        torch.save(model.state_dict(), "saved_models/"+op+"_"+str(idx)+".pth")
    del train_X_
    del train_graphs_

/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:205: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:206: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_iter = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
100%|██████████| 10/10 [44:43<00:00, 268.37s/it]
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:205: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_f

In [11]:
results = []
for idx in range(5):
    run_result = []
    for op in gt_operators:
        model = GraphTemporal(num_ch=19, num_t=NUM_WINDOWS, op=op).to(DEVICE)
        model.load_state_dict(torch.load("saved_models/"+op+"_"+str(idx)+".pth")) 
        test_res = print_acc(model, test_iter)
        run_result.append(test_res)
    results.append(run_result)

acc = ["accuracy", "f1", "precision", "recall"]
results = np.array(results).reshape(5, 5, 4)
for idx, res in enumerate(results):
    mean = np.mean(res, axis=0)
    std = np.std(res, axis=0)
    for mi, _ in enumerate(mean):
        print(gt_operators[idx], acc[mi], mean[mi], std[mi])

DCRNN accuracy 0.8130434782608695 0.018130428990140293
DCRNN f1 0.7901009940036658 0.017123654468809806
DCRNN precision 0.7698412698412698 0.024590370452110582
DCRNN recall 0.8133667360069692 0.03975105265997842
A3TGCN accuracy 0.8137681159420291 0.02263840485770046
A3TGCN f1 0.790187644510366 0.02125938695635057
A3TGCN precision 0.7666666666666666 0.027766437594501415
A3TGCN recall 0.8175647099201887 0.044899463879279115
GCLSTM accuracy 0.7963768115942029 0.011084825029549546
GCLSTM f1 0.7816069752166166 0.00605615776047297
GCLSTM precision 0.7984126984126985 0.043062412591271526
GCLSTM recall 0.7693140603712032 0.03686175286745506
TGCN accuracy 0.7891304347826087 0.02955953341213211
TGCN f1 0.774545400963142 0.021281729200443935
TGCN precision 0.7904761904761904 0.0562541115767281
TGCN recall 0.7689526825988076 0.07644227593631181
GConvGRU accuracy 0.8021739130434783 0.02516486283215268
GConvGRU f1 0.7876553146043245 0.009667642303717205
GConvGRU precision 0.8 0.05575926371508028
GCo

In [5]:
"""PLV connectivity"""
train_X = np.load("../tuh_data/train_X.npy", mmap_mode="c").astype(np.float16)
train_y = np.load("../tuh_data/train_y.npy", mmap_mode="c").astype(np.float16)
train_graphs = np.load("../tuh_data/train_graphs_plv.npy", mmap_mode="c").astype(np.float16)
test_y = np.load("../tuh_data/test_y.npy", mmap_mode="c").astype(np.float16)
test_X = np.load("../tuh_data/test_X.npy", mmap_mode="c").astype(np.float16)
test_graphs = np.load("../tuh_data/test_graphs_plv.npy", mmap_mode="c").astype(np.float16)
train_X = np.moveaxis(train_X, 1, 2)
test_X = np.moveaxis(test_X, 1, 2)

train_graphs, test_graphs = norm_adj(train_graphs, test_graphs)
train_iter, test_iter  = loaders(train_X, train_graphs, train_y, test_X, test_graphs, test_y, DEVICE, BATCH_SIZE, NUM_WINDOWS)

/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:172: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:278.)
  y = torch.tensor([y], dtype=torch.float).to(device)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:205: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:206: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_iter = DataLoader(test_dataset, 

In [6]:
for idx in range(5): #run 5 times
    train_X_, val_X, _, _ = train_test_split(train_X, train_y, random_state=idx, test_size=0.05)
    train_graphs_, val_graphs, train_y_, val_y = train_test_split(train_graphs, train_y, random_state=idx, test_size=0.05)
    train_iter, val_iter  = loaders(train_X_, train_graphs_, train_y_, val_X, val_graphs, val_y, DEVICE, BATCH_SIZE, NUM_WINDOWS)
    gt_operators = ["DCRNN", "A3TGCN", "GCLSTM", "TGCN", "GConvGRU"]
    for op in gt_operators:
        model = GraphTemporal(num_ch=19, num_t=NUM_WINDOWS, op=op).to(DEVICE)
        model = train_model(model, NUM_EPOCHS, train_iter, val_iter)
        torch.save(model.state_dict(), "saved_models/"+op+"_"+str(idx)+"_PLV.pth")
    del train_X_
    del train_graphs_

/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:205: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:206: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_iter = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
100%|██████████| 10/10 [42:33<00:00, 255.30s/it]
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_features.py:205: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
/Users/mohamedr/projects/limited_data/graphs/bands/dynamic_graphs/clean_code/clean_clean_code/features/make_f

In [9]:
results = []
for idx in range(5):
    run_result = []
    for op in gt_operators:
        model = GraphTemporal(num_ch=19, num_t=NUM_WINDOWS, op=op).to(DEVICE)
        model.load_state_dict(torch.load("saved_models/"+op+"_"+str(idx)+"_PLV.pth")) 
        test_res = print_acc(model, test_iter)
        run_result.append(test_res)
    results.append(run_result)

acc = ["accuracy", "f1", "precision", "recall"]
results = np.array(results).reshape(5, 5, 4)
for idx, res in enumerate(results):
    mean = np.mean(res, axis=0)
    std = np.std(res, axis=0)
    for mi, _ in enumerate(mean):
        print(gt_operators[idx], acc[mi], mean[mi], std[mi])

DCRNN accuracy 0.8014492753623188 0.019765480720269336
DCRNN f1 0.7892262287504851 0.014690099265645922
DCRNN precision 0.8126984126984127 0.040343698498853456
DCRNN recall 0.7714677558436186 0.05059905838579025
A3TGCN accuracy 0.805072463768116 0.02374778204083178
A3TGCN f1 0.7880097006187915 0.024023165091099842
A3TGCN precision 0.7936507936507937 0.04892392065848395
A3TGCN recall 0.7867637660135484 0.047959738891944374
GCLSTM accuracy 0.8086956521739129 0.019897869880791286
GCLSTM f1 0.7959938909490036 0.014751207996447437
GCLSTM precision 0.8158730158730159 0.03382107262327381
GCLSTM recall 0.7802347824450041 0.04331502083059288
TGCN accuracy 0.7927536231884057 0.025455316043840992
TGCN f1 0.7701822837663238 0.04077997925030473
TGCN precision 0.7746031746031746 0.11562269112047183
TGCN recall 0.7907273775200443 0.0800209117000193
GConvGRU accuracy 0.8 0.027820802638336934
GConvGRU f1 0.7817904692937951 0.010033000642006929
GConvGRU precision 0.7809523809523811 0.06856701925858785
G